# OCR — зургаас бодлого таних (FR-03, NFR-05 ≥ 85%)
Luna төслийн `ml/ml/ocr/` кодыг Colab GPU дээр ажиллуулж, үнэлж, pix2tex-ийг өөрийн зураг дээр fine-tune хийнэ.

**Runtime → Change runtime type → GPU (T4)** сонгоно.

| Алхам | Юу хийх |
|---|---|
| 1 | Орчин: repo, pix2tex, Tesseract (mon) |
| 2 | Drive-ийн зургийн хавтсыг холбох (`capstone_ocr/images/`) |
| 3 | Нэг зураг дээр турших |
| 4 | **Шошголох туслах**: `.tex`-гүй зургуудад pix2tex-ийн ноорог үүсгэх → гараар засах |
| 5 | Суурь (fine-tune хийгээгүй) үнэлгээ → `ocr_metrics.json` |
| 6 | Бодлогын сангаас **синтетик** томьёоны зураг үүсгэх (бодит зураг цуглуулж дуустал) |
| 7 | pix2tex fine-tune |
| 8 | Fine-tune хийсэн загварын үнэлгээ, жинг Drive-д хадгалах |

Drive-ийн бүтэц (нэг удаа үүсгэнэ):
```
MyDrive/capstone_ocr/
  images/      img_0001.jpg + img_0001.tex   ← бүтэн бодлогын зураг, .tex = бүх томьёо зайгаар
  crops/       img_0001_r0.png + .tex        ← томьёоны муж (4-р алхам үүсгэнэ), fine-tune-д
  synthetic/   syn_0001.png + .tex           ← 6-р алхам
  models/      weights.pth, config.yaml      ← 7-р алхам
  reports/     ocr_metrics_*.json
```

## 1. Орчин

In [ ]:
import os, subprocess, sys
from getpass import getpass

REPO = "github.com/ХЭРЭГЛЭГЧ/Luna"   # ← өөрийн GitHub репо: github.com/<username>/<repo>
BRANCH = "main"

if not os.path.exists("/content/Luna"):
    token = getpass("GitHub token (public репо бол хоосон Enter): ").strip()
    url = f"https://{'x-access-token:' + token + '@' if token else ''}{REPO}.git"
    r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "/content/Luna"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        # token-ийг хэвлэхгүй, зөвхөн git-ийн шалтгааныг харуулна
        err = r.stderr.replace(token, "***") if token else r.stderr
        raise SystemExit("git clone амжилтгүй:\n" + err +
                         "\nШалгах: репо нэр зөв үү? push хийгдсэн үү? салбар нэр? token-д энэ репо + Contents: Read сонгосон уу?")
%cd /content/Luna/ml
!git log --oneline -1

In [ ]:
# pix2tex + Tesseract (монгол хэл). Torch Colab-д суусан байдаг.
!pip -q install "pix2tex[train]>=0.1.2" pytesseract Pillow munch pyyaml
!apt-get -qq install -y tesseract-ocr > /dev/null
# Pipeline нь mon.traineddata-г data/raw/tesseract/-ээс хайдаг
!mkdir -p data/raw/tesseract && wget -q -O data/raw/tesseract/mon.traineddata \
    https://github.com/tesseract-ocr/tessdata/raw/main/mon.traineddata
!tesseract --version | head -1
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "АЛГА — Runtime → GPU сонго")

## 2. Drive холбох

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive/capstone_ocr")
for d in ("images", "crops", "synthetic", "models", "reports"):
    (DRIVE / d).mkdir(parents=True, exist_ok=True)

# evaluate.py нь data/own_images/-ийг уншдаг → Drive-ийн images/ руу холбоно
own = Path("data/own_images")
if own.is_symlink() or own.exists():
    own.unlink() if own.is_symlink() else None
own.symlink_to(DRIVE / "images")

imgs = sorted(p for p in (DRIVE/"images").iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg"})
labeled = [p for p in imgs if p.with_suffix(".tex").exists()]
print(f"Зураг: {len(imgs)}, .tex-тэй: {len(labeled)}, шошгогүй: {len(imgs)-len(labeled)}  (зорилт ≥ 300)")

## 3. Нэг зураг дээр турших

In [ ]:
import sys; sys.path.insert(0, "/content/Luna/ml")
from ml.ocr.pipeline import recognize
from IPython.display import display
from PIL import Image

if imgs:
    sample = imgs[0]
    display(Image.open(sample).resize((600, int(600 * Image.open(sample).height / Image.open(sample).width))))
    out = recognize(sample)
    print("Текст:", out["text"])
    print("LaTeX:", out["latex"])
else:
    print("images/ хоосон — утсаар авсан зургуудаа Drive-д хуулна уу.")

## 4. Шошголох туслах (ноорог `.tex` үүсгэх)
300 зургийг гараар LaTeX бичих нь удаан. Энэ алхам шошгогүй зураг бүрт:
- бүтэн зургийн ноорог `.tex` (pix2tex-ийн таамаг) — **та зөв болгож засна**
- томьёоны мужийн тайрмал `crops/img_0001_r0.png` + ноорог `.tex` — fine-tune-ийн өгөгдөл

Үүний дараа Drive дээр `.tex` файлуудыг нээж засаад, дараагийн алхмуудыг ажиллуулна.
`labels_review.csv`-д зураг тус бүрийн ноорог байгаа тул Excel-ээр ч засаж болно.

In [ ]:
import csv, re
import pytesseract
from ml.ocr import pipeline as P

rows = []
for img_path in imgs:
    tex = img_path.with_suffix(".tex")
    if tex.exists():
        continue
    img = Image.open(img_path).convert("RGB")
    data = pytesseract.image_to_data(img, lang="mon", config=P._tesseract_config(), output_type=pytesseract.Output.DICT)
    words = [{"text": data["text"][i], "line": (data["block_num"][i], data["par_num"][i], data["line_num"][i]),
              "left": data["left"][i], "top": data["top"][i], "width": data["width"][i], "height": data["height"][i]}
             for i in range(len(data["text"]))]
    _, regions = P.split_words(words)
    model = P._latex_model()
    latex = []
    for k, (x0, y0, x1, y1) in enumerate(regions or [(0, 0, img.width, img.height)]):
        crop = img.crop((max(0, x0-P.PAD), max(0, y0-P.PAD), min(img.width, x1+P.PAD), min(img.height, y1+P.PAD)))
        pred = model(crop)
        latex.append(pred)
        crop_path = DRIVE / "crops" / f"{img_path.stem}_r{k}.png"
        crop.save(crop_path)
        crop_path.with_suffix(".tex").write_text(pred, encoding="utf-8")
    draft = " ".join(latex)
    tex.write_text(draft, encoding="utf-8")
    rows.append({"image": img_path.name, "draft_latex": draft, "n_regions": len(regions)})

with open(DRIVE / "labels_review.csv", "a", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["image", "draft_latex", "n_regions"])
    if f.tell() == 0: w.writeheader()
    w.writerows(rows)
print(f"{len(rows)} зурагт ноорог .tex үүсгэв. Drive дээр засаад 5-р алхмыг ажиллуул.")

## 5. Суурь үнэлгээ (fine-tune хийгээгүй pix2tex)

In [ ]:
import json, shutil, datetime
!python -m ml.ocr.evaluate
if Path("reports/ocr_metrics.json").exists():
    m = json.loads(Path("reports/ocr_metrics.json").read_text())
    stamp = datetime.date.today().isoformat()
    shutil.copy("reports/ocr_metrics.json", DRIVE / "reports" / f"ocr_metrics_baseline_{stamp}.json")
    print(m)
    print("Exact match ≥ 0.85:", "✅" if m["exact_match"] >= 0.85 else "❌", "| char acc:", round(m["char_accuracy"], 3))

## 6. Синтетик өгөгдөл (сонголттой)
Бодлогын сангийн 60 LaTeX томьёог matplotlib-ээр зураг болгоно — бодит зураг цуглуулж дуустал
fine-tune-ийг турших, мөн сургалтын багцыг томруулахад. Бодит гар утасны зургийг орлохгүй
(гэрэл, өнцөг, хэвлэлийн фонт өөр) тул үнэлгээг **зөвхөн бодит зураг** дээр хийнэ.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd, random
random.seed(42)

bank = pd.read_csv("csv/problems.csv", encoding="utf-8-sig")
formulas = [s for s in bank["томьёо_LaTeX"].dropna().astype(str) if s.strip()]
ok, bad = 0, []
for i, f in enumerate(formulas, 1):
    for v in range(3):  # хэмжээ/фонтын 3 хувилбар
        try:
            fig = plt.figure(figsize=(4, 1), dpi=random.choice([120, 160, 200]))
            fig.text(0.05, 0.4, f"${f}$", fontsize=random.choice([14, 18, 22]),
                     family=random.choice(["serif", "DejaVu Sans"]))
            out = DRIVE / "synthetic" / f"syn_{i:04d}_{v}.png"
            fig.savefig(out, bbox_inches="tight", pad_inches=0.15, facecolor="white")
            plt.close(fig)
            out.with_suffix(".tex").write_text(f, encoding="utf-8")
            ok += 1
        except Exception as e:  # mathtext бүх LaTeX-ийг дэмждэггүй
            plt.close("all"); bad.append((f, str(e)[:60]))
print(f"Үүсгэсэн: {ok}, алгассан: {len(bad)}")
for f, e in bad[:5]: print("  ✗", f, "—", e)

## 7. pix2tex fine-tune
Сургалтын өгөгдөл = `crops/` (бодит, засагдсан `.tex`) + `synthetic/`. Зураг бүр **зөвхөн томьёо** агуулна.
Файлаар 85/15 хуваана (seed 42). Эхний удаа `EPOCHS=5`-аар туршаад, ажиллаж байвал нэмнэ.

> pix2tex-ийн сургалтын CLI хувилбараас хамаарч өөрчлөгдөж болно: алдаа гарвал
> https://github.com/lukas-blecher/LaTeX-OCR#training-the-model хэсэгтэй тулгана.

In [ ]:
import importlib.resources as ir, yaml, glob, shutil, random
random.seed(42)
EPOCHS, BATCH = 5, 16

pairs = []
for d in ("crops", "synthetic"):
    for img in sorted((DRIVE / d).glob("*.png")):
        tex = img.with_suffix(".tex")
        if tex.exists() and tex.read_text(encoding="utf-8").strip():
            pairs.append((img, tex.read_text(encoding="utf-8").strip()))
random.shuffle(pairs)
n_val = max(1, int(len(pairs) * 0.15))
splits = {"val": pairs[:n_val], "train": pairs[n_val:]}
print({k: len(v) for k, v in splits.items()})
assert len(splits["train"]) >= 20, "Сургалтын зураг хэт цөөн (≥20 хэрэгтэй) — 4 эсвэл 6-р алхмыг ажиллуул."

# pix2tex-ийн dataset формат: images/0000000.png ... + equations.txt (мөр = индекс)
work = Path("/content/p2t"); shutil.rmtree(work, ignore_errors=True)
for name, items in splits.items():
    (work / name / "images").mkdir(parents=True)
    with open(work / name / "equations.txt", "w", encoding="utf-8") as f:
        for i, (img, tex) in enumerate(items):
            Image.open(img).convert("RGB").save(work / name / "images" / f"{i:07d}.png")
            f.write(tex.replace("\n", " ") + "\n")

# Багц доторх анхдагч config + бэлэн жин (үүнээс үргэлжлүүлж сургана)
pkg = Path(str(ir.files("pix2tex")))
base_cfg = yaml.safe_load((pkg / "model" / "settings" / "config.yaml").read_text())
from pix2tex.cli import LatexOCR; LatexOCR()  # жинг татна
weights = sorted(glob.glob(str(pkg / "model" / "checkpoints" / "*.pth")))
tokenizer = str(pkg / "model" / "dataset" / "tokenizer.json")
print("Эхлэх жин:", weights[-1] if weights else "ОЛДСОНГҮЙ")

for name in ("train", "val"):
    !python -m pix2tex.dataset.dataset --equations {work/name}/equations.txt --images {work/name}/images \
        --tokenizer {tokenizer} --out {work/name}.pkl

cfg = {**base_cfg, "data": str(work/"train.pkl"), "valdata": str(work/"val.pkl"), "tokenizer": tokenizer,
       "load_chkpt": weights[-1] if weights else None, "epochs": EPOCHS, "batchsize": BATCH,
       "model_path": str(work/"checkpoints"), "name": "luna_pix2tex", "wandb": False, "debug": False,
       "save_freq": 1, "sample_freq": 500, "testbatchsize": 8, "valbatches": 50}
(work / "config.yaml").write_text(yaml.safe_dump(cfg))
!python -m pix2tex.train --config {work}/config.yaml

## 8. Fine-tune хийсэн загварын үнэлгээ ба хадгалах

In [ ]:
ckpts = sorted(glob.glob(str(work / "checkpoints" / "**" / "*.pth"), recursive=True), key=os.path.getmtime)
assert ckpts, "Checkpoint олдсонгүй — 7-р алхам амжилттай дууссан эсэхийг шалга."
best = ckpts[-1]
shutil.copy(best, DRIVE / "models" / "weights.pth")
shutil.copy(work / "config.yaml", DRIVE / "models" / "config.yaml")
print("Хадгалсан:", DRIVE / "models")

# Pipeline-д шинэ жинг орчны хувьсагчаар өгч, бодит зураг дээр дахин үнэлнэ
os.environ["PIX2TEX_CHECKPOINT"] = str(DRIVE / "models" / "weights.pth")
os.environ["PIX2TEX_CONFIG"] = str(DRIVE / "models" / "config.yaml")
!PIX2TEX_CHECKPOINT=$PIX2TEX_CHECKPOINT PIX2TEX_CONFIG=$PIX2TEX_CONFIG python -m ml.ocr.evaluate
m = json.loads(Path("reports/ocr_metrics.json").read_text())
shutil.copy("reports/ocr_metrics.json", DRIVE / "reports" / f"ocr_metrics_finetuned_{datetime.date.today().isoformat()}.json")
print(m)

## Дараа нь (локал / сервер дээр)
1. `MyDrive/capstone_ocr/models/{weights.pth,config.yaml}` → `Luna/ml/models/pix2tex/` (git-д орохгүй)
2. Ажиллуулахдаа:
   ```bash
   PIX2TEX_CHECKPOINT=ml/models/pix2tex/weights.pth PIX2TEX_CONFIG=ml/models/pix2tex/config.yaml \
     python -m ml.ocr.evaluate
   ```
3. `reports/results.md`-ийн OCR хэсгийг `evaluate.py` өөрөө шинэчилнэ — тоог **зохиохгүй**, зөвхөн энд гарсан утгыг тайланд бичнэ.
4. Exact match < 85% бол: зураг нэмэх (ялангуяа бодит гар утасны), EPOCHS нэмэх, `split_words`-ийн томьёоны муж илрүүлэлтийг сайжруулах (кирилл/математик тэмдэгтийн дүрэм).